# 04 — Признаки и обучение

**Финальная сборка.** Ноутбук собирает признаки, обучает модель на всех доступных
данных и пишет файл для отправки в Kaggle.

Прогон занимает несколько минут; фактический пик памяти печатает ячейка обучения.
Предыдущий ноутбук `03-feature-analysis.ipynb` разбирал, как эти признаки
искались и что получилось. Здесь сборка, обучение HGB, запись submission и
CV-only диагностика LightGBM; финальная модель и протокол не меняются.
Вариант и набор колонок читаются из `data/processed/03_summary.json`; вручную они
не задаются.

**Как получена локальная оценка и что она не измеряет.** Финальное обучение
запускается на всех 3 млн заявок, `early_stopping=False`; 300 итераций — это
фиксированный бюджет, а не проверка сходимости. Меток на test нет, поэтому
собственной локальной оценки у финальной модели нет: train-метрика была бы
in-sample по построению. Локальная оценка вне выборки для модели из `03` —
холдаут `[2 000 000, 3 000 000)`: **0.739142 ± 0.010409** (ROC AUC, среднее и
стандартное отклонение по восьми блокам). Холдаут не участвовал в выборе
признаков и конфигурации; это оценка модели из `03`, обученной на 2 млн заявок,
а не измерение качества финальной модели на 3 млн или на test. Числа
0.75478 / 0.74890 — внешний бейзлайн Kaggle (public / private), не наша метрика.


In [1]:
from __future__ import annotations

import gc
import hashlib
import json
import resource
import sys
import time
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
import pyarrow
import sklearn
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score


def find_repo_root() -> Path:
    """Ищет корень репозитория по pyproject.toml: ноутбук работает из любого каталога внутри репозитория."""
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise RuntimeError("Не найден корень репозитория: нет pyproject.toml вверх по дереву")


REPO_ROOT = find_repo_root()


def rel(path: Path) -> str:
    """Путь относительно корня репозитория: чтобы в выводах не печатался абсолютный путь."""
    try:
        return str(Path(path).resolve().relative_to(REPO_ROOT))
    except ValueError:
        return Path(path).name


def part_key(path: Path) -> tuple[int, str]:
    digits = "".join(character for character in path.stem if character.isdigit())
    return (int(digits) if digits else -1, path.name)


def peak_memory_gb() -> float:
    peak = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    return peak / 2**30 if sys.platform == "darwin" else peak / 2**20


def fmt_int(value: int) -> str:
    """Число с пробелами между тысячами; запятые в тексте не трогает."""
    return f"{value:,}".replace(",", " ")


DATA_DIR = REPO_ROOT / "data" / "raw" / "data_for_competition"
PROCESSED_DIR = REPO_ROOT / "data" / "processed"
TRAIN_PARTS = sorted((DATA_DIR / "train_data").glob("train_data_*.pq"), key=part_key)
TEST_PARTS = sorted((DATA_DIR / "test_data").glob("test_data_*.pq"), key=part_key)

SUMMARY_PATH = PROCESSED_DIR / "03_summary.json"
if not SUMMARY_PATH.is_file():
    raise FileNotFoundError(
        "Нет data/processed/03_summary.json: сначала выполните "
        "notebooks/03-feature-analysis.ipynb"
    )
SUMMARY = json.loads(SUMMARY_PATH.read_text(encoding="utf-8"))
FINAL_VARIANT = SUMMARY["variant"]
FINAL_FEATURES = int(SUMMARY["features"])
FINAL_COLUMNS = list(SUMMARY["columns"])
HOLDOUT_MEAN = float(SUMMARY["holdout_mean"])
HOLDOUT_SD = float(SUMMARY["holdout_sd"])
CV_MEAN = float(SUMMARY["cv_mean"])
assert len(FINAL_COLUMNS) == FINAL_FEATURES
assert SUMMARY["holdout_blocks"] == 8

ALPHA = 200.0
MIN_LEVEL_ROWS = 2000
WINDOW = 3
SEED = 42

HGB_PARAMS = dict(
    max_iter=300,
    learning_rate=0.08,
    min_samples_leaf=200,
    l2_regularization=1.0,
    early_stopping=False,
    random_state=SEED,
)

CV_END = 2_000_000
PART_SIZE = 250_000
FOLD_FIRST, FOLD_LAST = 2, 7
COMPARISON_PATH = PROCESSED_DIR / "04_model_comparison.json"

LGBM_PARAMS = dict(
    objective="binary",
    n_estimators=300,
    learning_rate=0.08,
    num_leaves=31,
    min_child_samples=200,
    reg_lambda=1.0,
    random_state=SEED,
    n_jobs=-1,
    verbosity=-1,
)

assert FINAL_VARIANT == "without_first"
assert FINAL_FEATURES == 79
assert len(FINAL_COLUMNS) == 79
assert FOLD_LAST - FOLD_FIRST + 1 == 6
assert CV_END == 8 * PART_SIZE
print(f"CV из 03: {CV_MEAN:.6f}; сравнение только на id < {CV_END:,}".replace(",", " "))

RISK_COLUMNS = [
    "pre_over2limit",
    "pre_util",
    "pre_loans_credit_cost_rate",
    "enc_loans_credit_status",
    "enc_loans_credit_type",
    "pre_maxover2limit",
    "pre_loans_credit_limit",
    "pre_since_opened",
    "pre_loans_next_pay_summ",
    "pre_loans_outstanding",
    "pre_loans_max_overdue_sum",
    "pre_pterm",
    "pre_fterm",
    "pre_till_pclose",
    "pre_till_fclose",
    "pre_since_confirmed",
]
PAYM_COLUMNS = [f"enc_paym_{index}" for index in range(25)]
ENCODED_COLUMNS = [*RISK_COLUMNS, *PAYM_COLUMNS]
ZERO_COLUMNS = [
    "is_zero_loans5",
    "is_zero_loans530",
    "is_zero_loans3060",
    "is_zero_loans6090",
    "is_zero_loans90",
    "is_zero_util",
    "is_zero_over2limit",
    "is_zero_maxover2limit",
]
OVERDUE_BUCKETS = ["loans5", "loans530", "loans3060", "loans6090", "loans90"]
OVERDUE_WEIGHTS = [2.0, 3.0, 4.0, 6.0, 8.0]
READ_COLUMNS = ["id", "rn", *ENCODED_COLUMNS, *ZERO_COLUMNS, "fclose_flag", "pclose_flag"]

TRAIN_TARGET = pd.read_csv(DATA_DIR / "train_target.csv")
TEST_TARGET = pd.read_csv(DATA_DIR / "test_target.csv")
SAMPLE = pd.read_csv(DATA_DIR / "sample_submission.csv")
assert TRAIN_TARGET["id"].is_monotonic_increasing, "train_target.csv должен идти по возрастанию id"
assert len(TEST_TARGET) == 500_000
assert TEST_TARGET["id"].tolist() == SAMPLE["id"].tolist()
FLAGS = TRAIN_TARGET.set_index("id")["flag"]
assert len(FLAGS) == 3_000_000
assert FLAGS.notna().all()

print(f"данные:           {rel(DATA_DIR)}")
print(f"train-партиций:   {len(TRAIN_PARTS)}")
print(f"test-партиций:    {len(TEST_PARTS)}")
print(f"заявок train:     {fmt_int(len(TRAIN_TARGET))}")
print(f"заявок test:      {fmt_int(len(TEST_TARGET))}")
print(f"финальный вариант: {FINAL_VARIANT}, признаков: {FINAL_FEATURES}")
print(f"оценка из 03:     {HOLDOUT_MEAN:.6f} ± {HOLDOUT_SD:.6f} (ROC AUC, SD по 8 блокам)")
FEATURES_TOTAL = (
    3  # n_records, share_open, share_noplan
    + 1  # cold_start
    + len(ZERO_COLUMNS) * 2  # nz_<бакет> и nzshare_<бакет>
    + 5  # overdue_ladder, overdue_sev, overdue_sev530, any_overdue, any_overdue_last3
    + 1  # last_util_nz
    + len(RISK_COLUMNS) * 4  # mean, last3, last, first
    + 4  # RISK_mean, RISK_mean_all_max, RISK_max, RISK_last3
    + 5  # PAYM_mean, PAYM_last, PAYM_first, PAYM_wlast, PAYM_recency
)
assert FEATURES_TOTAL == 99
print(f"признаков до выбора: {FEATURES_TOTAL}")


CV из 03: 0.750282; сравнение только на id < 2 000 000


данные:           data/raw/data_for_competition
train-партиций:   12
test-партиций:    2
заявок train:     3 000 000
заявок test:      500 000
финальный вариант: without_first, признаков: 79
оценка из 03:     0.739142 ± 0.010409 (ROC AUC, SD по 8 блокам)
признаков до выбора: 99


## 1. OOF-кодировка уровней

Номера уровней в `pre_*` и `enc_*` присвоены случайно, поэтому уровень
кодируется сглаженным логарифмом шансов:

```
LO[c][l] = ln( (k_l + 200 · p) / (n_l − k_l + 200 · (1 − p)) )
```

Уровни, на которых меньше 2000 строк, обнуляются: оценка доли на них шумовая, и
нейтральный ноль не вносит сигнала, которого нет в данных, в отличие от оценки
на малой выборке.

Кодировка строится out-of-fold по времени: для партиции `k` берутся счётчики
партиций `0…k-1`, партиция 0 получает нейтральный ноль. Строка не видит
собственную метку, поэтому признаки train не in-sample. Для test используется
`FULL_ENCODER = ENCODERS[12]` — статистика всех 12 партиций: test идёт после
train, и это самый полный доступный словарь. В `03` холдаут кодировался
`ENCODERS[8]` (статистика `0..7`), потому что там нужно было воспроизвести момент
решения; здесь задача другая — предсказать будущие заявки, поэтому словарь
шире. Это разные словари; их числа напрямую не сравниваются.

Приор сглаживания `p` — строчная доля положительных строк по всем 12
партициям: счётчики уровней тоже строчные. Доля заявок `id < 2M` печатается
отдельно для сравнения; насколько смена приора меняет log-odds, показывает
ячейка ниже.

Счётчики накапливаются по одной партиции: так в памяти остаётся одна партиция, а не весь train.


In [2]:
def read_part(part: Path, columns: list[str] | None = None) -> pd.DataFrame:
    """Читает одну партицию в int32. Больше одной партиции за раз в памяти не держим."""
    return pd.read_parquet(part, columns=columns).astype(np.int32)


def grow(values: np.ndarray, size: int) -> np.ndarray:
    """Дополняет массив до нужной длины; не обрезает."""
    if len(values) >= size:
        return values
    return np.pad(values, (0, size - len(values)))


def application_bounds(ids: np.ndarray, rn: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Границы блоков строк по заявкам с полной проверкой rn == 1..n."""
    if np.any(np.diff(ids) < 0):
        raise ValueError("id внутри партиции не отсортированы")
    starts = np.flatnonzero(np.r_[True, np.diff(ids) != 0])
    ends = np.r_[starts[1:], len(ids)]
    lengths = ends - starts
    position = np.arange(len(ids)) - np.repeat(starts, lengths) + 1
    if not (rn == position).all():
        raise ValueError("rn не равен 1..n хотя бы у одной заявки")
    return starts, ends


if not TRAIN_PARTS:
    raise FileNotFoundError(
        f"В {DATA_DIR / 'train_data'} нет файлов train_data_*.pq. Одного "
        "train_target.csv не хватает: счётчики уровней считаются по партициям, "
        "без них цикл не выполнится и падение придёт на базовой доле."
    )

started = time.perf_counter()
PART_COUNTS: list[dict[str, np.ndarray]] = []
PART_POSITIVES: list[dict[str, np.ndarray]] = []
PART_ROWS: list[int] = []                 # строки истории
PART_POSITIVE: list[int] = []             # положительные строки истории
PART_APPLICATIONS: list[int] = []
PART_APPLICATION_POSITIVE: list[int] = []

for part in TRAIN_PARTS:
    frame = read_part(part, ["id", *ENCODED_COLUMNS])
    ids = frame["id"].to_numpy()
    labels = FLAGS.reindex(ids).to_numpy(dtype=np.float64)
    if np.isnan(labels).any():
        raise ValueError(f"в {part.name} есть id без метки")
    counts_part: dict[str, np.ndarray] = {}
    positives_part: dict[str, np.ndarray] = {}
    for column in ENCODED_COLUMNS:
        values = frame[column].to_numpy()
        size = int(values.max()) + 1
        counts_part[column] = np.bincount(values, minlength=size)
        positives_part[column] = np.bincount(values, weights=labels, minlength=size)
    PART_COUNTS.append(counts_part)
    PART_POSITIVES.append(positives_part)
    application_ids = pd.unique(ids)
    PART_ROWS.append(len(ids))
    PART_POSITIVE.append(int(labels.sum()))
    PART_APPLICATIONS.append(len(application_ids))
    PART_APPLICATION_POSITIVE.append(int(FLAGS.reindex(application_ids).sum()))
    del frame, ids, labels, application_ids
    _ = gc.collect()

assert len(PART_COUNTS) == len(PART_POSITIVES) == 12
assert len(PART_ROWS) == len(PART_POSITIVE) == len(PART_APPLICATIONS) == 12
assert sum(PART_APPLICATIONS) == 3_000_000
assert sum(PART_APPLICATIONS[:8]) == 2_000_000

BASE_RATE = sum(PART_POSITIVE) / sum(PART_ROWS)
APPLICATION_RATE_FIT = sum(PART_APPLICATION_POSITIVE[:8]) / sum(PART_APPLICATIONS[:8])
APPLICATION_RATE_ALL = sum(PART_APPLICATION_POSITIVE) / sum(PART_APPLICATIONS)
assert 0 < BASE_RATE < 1

print(f"счётчики {len(ENCODED_COLUMNS)} кодируемых колонок собраны за {time.perf_counter() - started:.0f}s")
print(f"строк истории (все 12 партиций): {fmt_int(sum(PART_ROWS))}")
print(f"положительных строк: {fmt_int(sum(PART_POSITIVE))}")
print(f"строчная доля положительных (BASE_RATE): {BASE_RATE:.4f} (доля строк)")
print(
    f"доля заявок id < 2M: {APPLICATION_RATE_FIT:.4f} (доля заявок; "
    f"{fmt_int(sum(PART_APPLICATION_POSITIVE[:8]))} из {fmt_int(sum(PART_APPLICATIONS[:8]))})"
)
print(
    f"доля заявок всех 12 партиций: {APPLICATION_RATE_ALL:.4f} (доля заявок; "
    f"{fmt_int(sum(PART_APPLICATION_POSITIVE))} из {fmt_int(sum(PART_APPLICATIONS))})"
)
print(
    "доля заявок по партициям: "
    + ", ".join(f"{PART_APPLICATION_POSITIVE[k] / PART_APPLICATIONS[k]:.4f}" for k in range(12))
)
print(
    "строчная доля по партициям: "
    + ", ".join(f"{PART_POSITIVE[k] / PART_ROWS[k]:.4f}" for k in range(12))
)
print(f"пик памяти: {peak_memory_gb():.2f} ГБ")


счётчики 41 кодируемых колонок собраны за 13s
строк истории (все 12 партиций): 26 162 717
положительных строк: 879 809
строчная доля положительных (BASE_RATE): 0.0336 (доля строк)
доля заявок id < 2M: 0.0362 (доля заявок; 72 320 из 2 000 000)
доля заявок всех 12 партиций: 0.0355 (доля заявок; 106 442 из 3 000 000)
доля заявок по партициям: 0.0309, 0.0328, 0.0352, 0.0364, 0.0407, 0.0372, 0.0385, 0.0375, 0.0377, 0.0331, 0.0337, 0.0319
строчная доля по партициям: 0.0289, 0.0298, 0.0325, 0.0339, 0.0375, 0.0351, 0.0361, 0.0360, 0.0367, 0.0328, 0.0323, 0.0316
пик памяти: 3.90 ГБ


In [3]:
def make_encoder(
    counts: dict[str, np.ndarray],
    positives: dict[str, np.ndarray],
    base_rate: float,
) -> dict[str, np.ndarray]:
    """Строит отображение уровень → log-odds с обнулением редких уровней."""
    encoder = {}
    for column, column_counts in counts.items():
        column_positives = positives[column][: len(column_counts)]
        odds = np.log(
            (column_positives + ALPHA * base_rate)
            / (column_counts - column_positives + ALPHA * (1 - base_rate))
        )
        odds[column_counts < MIN_LEVEL_ROWS] = 0.0
        encoder[column] = odds.astype(np.float32)
    return encoder


def build_encoders(part_counts, part_positives, part_rows, part_positive,
                   columns) -> list[dict[str, np.ndarray]]:
    """encoders[k] — по партициям 0..k-1; encoders[12] — по всем 12 партициям."""
    encoders, rows, positives = [], 0, 0
    total_counts = {c: np.zeros(0, dtype=np.int64) for c in columns}
    total_positives = {c: np.zeros(0, dtype=np.float64) for c in columns}
    for k in range(13):
        if k == 0:
            encoders.append({c: np.zeros(1, dtype=np.float32) for c in columns})
        else:
            encoders.append(make_encoder(total_counts, total_positives,
                                         positives / rows))
        if k < 12:
            for c in columns:
                counts = part_counts[k][c].astype(np.int64)
                total_counts[c] = grow(total_counts[c], len(counts))
                total_counts[c][: len(counts)] += counts
                pos = part_positives[k][c].astype(np.float64)
                total_positives[c] = grow(total_positives[c], len(pos))
                total_positives[c][: len(pos)] += pos
            rows += part_rows[k]
            positives += part_positive[k]
    return encoders


def sum_counts(parts: list[dict[str, np.ndarray]]) -> dict[str, np.ndarray]:
    """Сумма счётчиков по партициям; длины дополняются нулями до максимума."""
    total = {}
    for column in ENCODED_COLUMNS:
        size = max(len(part[column]) for part in parts)
        values = np.zeros(size, dtype=np.int64)
        for part in parts:
            counts = part[column]
            values[: len(counts)] += counts
        total[column] = values
    return total


def sum_positives(parts: list[dict[str, np.ndarray]]) -> dict[str, np.ndarray]:
    """Сумма положительных счётчиков; длины дополняются нулями до максимума."""
    total = {}
    for column in ENCODED_COLUMNS:
        size = max(len(part[column]) for part in parts)
        values = np.zeros(size, dtype=np.float64)
        for part in parts:
            positives = part[column]
            values[: len(positives)] += positives
        total[column] = values
    return total


ALL_COUNTS = sum_counts(PART_COUNTS)
ALL_POSITIVES = sum_positives(PART_POSITIVES)
ENCODERS = build_encoders(PART_COUNTS, PART_POSITIVES, PART_ROWS,
                          PART_POSITIVE, ENCODED_COLUMNS)
FULL_ENCODER = ENCODERS[12]

assert all((table == 0).all() for table in ENCODERS[0].values())
for column in ENCODED_COLUMNS:
    assert len(FULL_ENCODER[column]) == len(ALL_COUNTS[column])
for k in (1, 8, 12):
    expected = make_encoder(
        sum_counts(PART_COUNTS[:k]),
        sum_positives(PART_POSITIVES[:k]),
        sum(PART_POSITIVE[:k]) / sum(PART_ROWS[:k]),
    )
    for column in ENCODED_COLUMNS:
        assert np.array_equal(ENCODERS[k][column], expected[column]), (k, column)

print(
    "строчный prior по партициям (положительные строки / строки истории): "
    + ", ".join(f"{PART_POSITIVE[k] / PART_ROWS[k]:.4f}" for k in range(12))
)
print(
    "максимальная длина таблицы кодировщика: "
    + ", ".join(
        f"{k}:{max(len(ENCODERS[k][c]) for c in ENCODED_COLUMNS)}" for k in range(13)
    )
)
for column in ENCODED_COLUMNS:
    lengths = [len(PART_COUNTS[k][column]) for k in range(12)]
    assert lengths == sorted(lengths), column


def encode(column: str, values: np.ndarray, encoder: dict[str, np.ndarray]) -> np.ndarray:
    """Переводит уровни в log-odds.

    Уровень за пределами таблицы получает нейтральный ноль: он неизвестен
    модели, и выдумывать для него значение нельзя.
    """
    table = encoder[column]
    if values.size and int(values.max()) >= len(table):
        table = np.pad(table, (0, int(values.max()) + 1 - len(table)))
    return table[values]


def spread_of(column: str) -> float:
    visible = ALL_COUNTS[column] >= MIN_LEVEL_ROWS
    if not visible.any():
        return 0.0
    return float(np.ptp(FULL_ENCODER[column][visible]))


spread = pd.DataFrame(
    {
        "колонка": list(FULL_ENCODER),
        "уровней_с_поддержкой": [
            int((ALL_COUNTS[column] >= MIN_LEVEL_ROWS).sum()) for column in FULL_ENCODER
        ],
        "разброс_log_odds": [
            round(spread_of(column), 3) for column in FULL_ENCODER
        ],
    }
).sort_values("разброс_log_odds", ascending=False)
print("\nкодировки с наибольшим разбросом log-odds (FULL_ENCODER, все 12 партиций):")
print(spread.head(10).to_string(index=False))
zero_spread = int((spread["разброс_log_odds"] == 0).sum())
print(f"\nколонок с нулевым разбросом (информации нет): {zero_spread}")
application_prior_encoder = make_encoder(ALL_COUNTS, ALL_POSITIVES, APPLICATION_RATE_ALL)
prior_shift = max(
    float(np.max(np.abs(FULL_ENCODER[column] - application_prior_encoder[column])))
    for column in ENCODED_COLUMNS
)
print(
    f"\nприор: строчная доля {BASE_RATE:.4f}; доля заявок id < 2M {APPLICATION_RATE_FIT:.4f}; "
    f"доля заявок всех 12 {APPLICATION_RATE_ALL:.4f}; "
    f"максимальное изменение log-odds при смене приора: {prior_shift:.4f} (log-odds)"
)
_ = gc.collect()


строчный prior по партициям (положительные строки / строки истории): 0.0289, 0.0298, 0.0325, 0.0339, 0.0375, 0.0351, 0.0361, 0.0360, 0.0367, 0.0328, 0.0323, 0.0316
максимальная длина таблицы кодировщика: 0:1, 1:20, 2:20, 3:20, 4:20, 5:20, 6:20, 7:20, 8:20, 9:20, 10:20, 11:20, 12:20

кодировки с наибольшим разбросом log-odds (FULL_ENCODER, все 12 партиций):
                   колонка  уровней_с_поддержкой  разброс_log_odds
   enc_loans_credit_status                     6             1.809
                  pre_util                    20             1.653
pre_loans_credit_cost_rate                    13             1.448
     enc_loans_credit_type                     8             1.447
            pre_over2limit                     6             1.385
         pre_maxover2limit                    20             1.264
    pre_loans_credit_limit                    20             0.950
               enc_paym_18                     4             0.950
               enc_paym_20            

### Проверка на test

Уровни в test могут отсутствовать в train: проверяем, как часто это происходит.
Такие уровни получают нейтральный ноль. Считаем, сколько пар «строка ×
кодируемая колонка» получают нейтральный ноль — из-за уровня за пределами
таблицы или обнуления редких, — и сколько строк это затрагивает.

Аудит идёт против `FULL_ENCODER` (все 12 партиций). В `03` холдаут проверялся
против `ENCODERS[8]` (партиции `0..7`): там нужно было увидеть, что известно на
момент решения, здесь — что известно модели, которая предсказывает test. Это
разные срезы и словари; их числа напрямую не сравниваются.


In [4]:
rows = []
unknown_parts = []
rare_parts = []
unknown_example = None
rare_by_column = {column: 0 for column in ENCODED_COLUMNS}
pairs_total = 0
test_rows = 0
n_test_applications = 0

for part in TEST_PARTS:
    frame = read_part(part, ["id", "rn", *ENCODED_COLUMNS])
    ids = frame["id"].to_numpy()
    rns = frame["rn"].to_numpy()
    starts, ends = application_bounds(ids, rns)
    part_unknown = np.zeros(len(frame), dtype=bool)
    part_rare = np.zeros(len(frame), dtype=bool)
    for column in ENCODED_COLUMNS:
        values = frame[column].to_numpy()
        table_length = len(FULL_ENCODER[column])
        known = values < table_length
        rare = known & (ALL_COUNTS[column][np.where(known, values, 0)] < MIN_LEVEL_ROWS)
        if unknown_example is None and not bool(known.all()):
            position = int(np.argmax(~known))
            unknown_example = (
                f"id = {int(ids[position])}, rn = {int(rns[position])}, "
                f"колонка = {column}, уровень = {int(values[position])}, "
                f"длина таблицы = {table_length}"
            )
        rare_by_column[column] += int(rare.sum())
        part_unknown |= ~known
        part_rare |= rare
        rows.append(
            {
                "колонка": column,
                "часть": part.name,
                "строк": len(values),
                "неизвестных_уровней": int((~known).sum()),
                "доля_неизвестных": round(float((~known).mean()), 6),
            }
        )
    unknown_parts.append(part_unknown)
    rare_parts.append(part_rare)
    pairs_total += len(frame) * len(ENCODED_COLUMNS)
    test_rows += len(frame)
    n_test_applications += len(starts)
    del frame, ids, rns, starts, ends
    _ = gc.collect()

assert n_test_applications == 500_000

unseen = pd.DataFrame(rows)
unseen = unseen.groupby("колонка", as_index=False).agg(
    строк=("строк", "sum"),
    неизвестных=("неизвестных_уровней", "sum"),
)
unseen["доля"] = (unseen["неизвестных"] / unseen["строк"]).round(6)
unseen = unseen.sort_values("неизвестных", ascending=False)
print("строки test с уровнями, которых не было в train (против FULL_ENCODER, все 12 партиций):")
print(unseen.head(10).to_string(index=False))
print(
    f"\nколонок, где таких строк нет вовсе: "
    f"{int((unseen['неизвестных'] == 0).sum())} из {len(unseen)}"
)

unknown_mask = np.concatenate(unknown_parts)
rare_mask = np.concatenate(rare_parts)
neutral_mask = unknown_mask | rare_mask
neutral_rows = int(neutral_mask.sum())
unknown_pairs = int(unseen["неизвестных"].sum())
rare_pairs = int(sum(rare_by_column.values()))
neutral_pairs = unknown_pairs + rare_pairs
print(f"\nстрок test:                         {fmt_int(test_rows)}")
print(f"всего пар (строка × колонка):       {fmt_int(pairs_total)}")
print(f"пар с неизвестным уровнем:          {fmt_int(unknown_pairs)}")
print(f"пар на редких известных уровнях:    {fmt_int(rare_pairs)}")
print(f"пар с нейтральным нулём:            {fmt_int(neutral_pairs)}")
print(f"доля нейтрального нуля от пар: {100 * neutral_pairs / pairs_total:.4f} %")
print(
    f"\nразличных строк с нейтральным нулём: {fmt_int(neutral_rows)}; "
    f"с неизвестным уровнем — {fmt_int(int(unknown_mask.sum()))}; "
    f"на редких известных уровнях — {fmt_int(int(rare_mask.sum()))}"
)
print(f"\nпример строки с неизвестным уровнем: {unknown_example}")
rare_table = pd.DataFrame(
    {
        "колонка": list(rare_by_column),
        "строк_на_редких": list(rare_by_column.values()),
    }
).sort_values("строк_на_редких", ascending=False)
print("\nтоп-колонки по строкам на редких известных уровнях:")
print(rare_table.head(5).to_string(index=False))
_ = gc.collect()


строки test с уровнями, которых не было в train (против FULL_ENCODER, все 12 партиций):
                   колонка   строк  неизвестных  доля
   pre_loans_next_pay_summ 4724601            1   0.0
                enc_paym_4 4724601            0   0.0
                enc_paym_6 4724601            0   0.0
                enc_paym_7 4724601            0   0.0
                enc_paym_8 4724601            0   0.0
                enc_paym_9 4724601            0   0.0
                 pre_fterm 4724601            0   0.0
pre_loans_credit_cost_rate 4724601            0   0.0
    pre_loans_credit_limit 4724601            0   0.0
 pre_loans_max_overdue_sum 4724601            0   0.0

колонок, где таких строк нет вовсе: 40 из 41

строк test:                         4 724 601
всего пар (строка × колонка):       193 708 641
пар с неизвестным уровнем:          1
пар на редких известных уровнях:    5 018
пар с нейтральным нулём:            5 019
доля нейтрального нуля от пар: 0.0026 %

различных стро

## 2. Агрегация до уровня заявки

Кредитная история — это 26 млн строк по 3 млн заявок, в среднем 8.7
записи на заявку. Признаки считаются по заявке, поэтому строки каждой партиции
сворачиваются в агрегаты.

Окно последних трёх записей, первая запись, последняя запись и общие средние —
по всей истории. Каждая партиция содержит 250 000 заявок и всю историю каждой
из них, поэтому результат склеивается построчно.


In [5]:
def segment_sum(values: np.ndarray, starts: np.ndarray) -> np.ndarray:
    return np.add.reduceat(values.astype(np.float32), starts)


def tail_sum(values: np.ndarray, starts: np.ndarray, ends: np.ndarray) -> np.ndarray:
    lengths = ends - starts
    tail_start = np.repeat(np.maximum(ends - WINDOW, starts), lengths)
    mask = np.arange(len(values)) >= tail_start
    return segment_sum(np.where(mask, values, np.float32(0.0)), starts)


def build_part(part: Path, encoder: dict[str, np.ndarray], cold_start: bool) -> pd.DataFrame:
    """Признаки одной заявки, индекс — id заявки."""
    frame = read_part(part, READ_COLUMNS)
    ids = frame["id"].to_numpy()
    starts, ends = application_bounds(ids, frame["rn"].to_numpy())
    sizes = (ends - starts).astype(np.float32)
    window_denominator = np.minimum(sizes, np.float32(WINDOW))

    out: dict[str, np.ndarray] = {
        "n_records": sizes,
        "cold_start": np.full(len(starts), np.float32(cold_start), dtype=np.float32),
    }
    out["share_open"] = segment_sum(frame["fclose_flag"].to_numpy() == 1, starts) / sizes
    out["share_noplan"] = segment_sum(frame["pclose_flag"].to_numpy() == 1, starts) / sizes

    for column in ZERO_COLUMNS:
        tag = column.removeprefix("is_zero_")
        nonzero = segment_sum(frame[column].to_numpy() == 0, starts)
        out[f"nz_{tag}"] = nonzero
        out[f"nzshare_{tag}"] = nonzero / sizes

    ladder = np.zeros(len(frame), dtype=np.float32)
    severity = np.zeros(len(frame), dtype=np.float32)
    weighted = np.zeros(len(frame), dtype=np.float32)
    any_overdue = np.zeros(len(frame), dtype=bool)
    for position, tag in enumerate(OVERDUE_BUCKETS, start=1):
        overdue = frame[f"is_zero_{tag}"].to_numpy() == 0
        any_overdue |= overdue
        ladder += np.float32(position) * overdue
        if position > 1:
            severity += np.float32(position - 1) * overdue
        weighted += np.float32(OVERDUE_WEIGHTS[position - 1]) * overdue
    out["overdue_ladder"] = segment_sum(ladder, starts)
    out["overdue_sev"] = segment_sum(severity, starts)
    out["overdue_sev530"] = segment_sum(weighted, starts)
    out["any_overdue"] = segment_sum(any_overdue, starts) / sizes
    out["any_overdue_last3"] = (
        tail_sum(any_overdue.astype(np.float32), starts, ends) / window_denominator
    )
    out["last_util_nz"] = (frame["is_zero_util"].to_numpy()[ends - 1] == 0).astype(np.float32)

    row_total = np.zeros(len(frame), dtype=np.float32)
    row_max = np.full(len(frame), -np.inf, dtype=np.float32)
    for column in RISK_COLUMNS:
        encoded = encode(column, frame[column].to_numpy(), encoder)
        row_total += encoded
        row_max = np.maximum(row_max, encoded)
        out[f"mean_{column}"] = segment_sum(encoded, starts) / sizes
        out[f"last3_{column}"] = tail_sum(encoded, starts, ends) / window_denominator
        out[f"last_{column}"] = encoded[ends - 1]
        out[f"first_{column}"] = encoded[starts]
    if not np.isfinite(row_max).all():
        raise ValueError("row_max содержит -inf: у заявки нет ни одной RISK-колонки")
    risk_count = np.float32(len(RISK_COLUMNS))
    out["RISK_mean"] = segment_sum(row_total, starts) / (sizes * risk_count)
    out["RISK_mean_all_max"] = segment_sum(row_max, starts) / sizes
    out["RISK_max"] = row_max[ends - 1]
    out["RISK_last3"] = tail_sum(row_total, starts, ends) / window_denominator / risk_count

    payment_sum = np.zeros(len(frame), dtype=np.float32)
    weighted_last = np.zeros(len(starts), dtype=np.float32)
    for index, column in enumerate(PAYM_COLUMNS, start=1):
        encoded = encode(column, frame[column].to_numpy(), encoder)
        payment_sum += encoded
        weighted_last += np.float32(1.0 / (1.0 + index)) * encoded[ends - 1]
    out["PAYM_mean"] = segment_sum(payment_sum, starts) / sizes
    out["PAYM_last"] = payment_sum[ends - 1]
    out["PAYM_first"] = payment_sum[starts]
    out["PAYM_wlast"] = weighted_last
    out["PAYM_recency"] = tail_sum(payment_sum, starts, ends) / window_denominator

    return pd.DataFrame(out, index=pd.Index(ids[starts], name="id"))


In [6]:
started = time.perf_counter()
train_parts = []
for k, part in enumerate(TRAIN_PARTS):
    train_parts.append(build_part(part, ENCODERS[k], cold_start=(k == 0)))
    print(
        f"{part.name}: {train_parts[-1].shape} за {time.perf_counter() - started:.0f}s, "
        f"пик {peak_memory_gb():.2f} ГБ"
    )
X_TRAIN = pd.concat(train_parts, axis=0)
ALL_COLUMNS = list(X_TRAIN.columns)
train_parts.clear()
_ = gc.collect()

test_parts = []
for part in TEST_PARTS:
    test_parts.append(build_part(part, FULL_ENCODER, cold_start=False))
    print(
        f"{part.name}: {test_parts[-1].shape} за {time.perf_counter() - started:.0f}s, "
        f"пик {peak_memory_gb():.2f} ГБ"
    )
X_TEST = pd.concat(test_parts, axis=0)
test_parts.clear()
_ = gc.collect()

assert X_TRAIN.shape == (3_000_000, 99)
assert X_TEST.shape == (500_000, 99)
assert len(X_TEST) == n_test_applications
assert len(ALL_COLUMNS) == FEATURES_TOTAL == 99
assert ALL_COLUMNS[:2] == ["n_records", "cold_start"]
assert not any(column.startswith("rawlast") for column in ALL_COLUMNS)
assert int(X_TRAIN["cold_start"].sum()) == 250_000
assert int(X_TEST["cold_start"].sum()) == 0
assert X_TRAIN.index.is_unique and X_TEST.index.is_unique
assert X_TRAIN.index.is_monotonic_increasing and X_TEST.index.is_monotonic_increasing
assert X_TRAIN.index.min() == 0 and X_TRAIN.index.max() == 2_999_999
assert X_TEST.index.min() == 3_000_000 and X_TEST.index.max() == 3_499_999
assert list(X_TEST.columns) == ALL_COLUMNS
assert not X_TRAIN.isna().any().any()
assert not X_TEST.isna().any().any()
assert np.isfinite(X_TRAIN.to_numpy()).all()
assert np.isfinite(X_TEST.to_numpy()).all()

print(f"\nX_TRAIN: {X_TRAIN.shape}, {X_TRAIN.memory_usage(deep=True).sum() / 2**30:.2f} ГБ")
print(f"X_TEST:  {X_TEST.shape}, {X_TEST.memory_usage(deep=True).sum() / 2**30:.2f} ГБ")
print(f"сборка: {time.perf_counter() - started:.0f}s, пик памяти {peak_memory_gb():.2f} ГБ")


train_data_0.pq: (250000, 99) за 1s, пик 4.40 ГБ


train_data_1.pq: (250000, 99) за 2s, пик 4.64 ГБ


train_data_2.pq: (250000, 99) за 3s, пик 4.69 ГБ


train_data_3.pq: (250000, 99) за 4s, пик 4.69 ГБ


train_data_4.pq: (250000, 99) за 5s, пик 4.76 ГБ


...

train_data_10.pq: (250000, 99) за 10s, пик 4.93 ГБ


train_data_11.pq: (250000, 99) за 12s, пик 4.93 ГБ


test_data_0.pq: (250000, 99) за 13s, пик 5.26 ГБ


test_data_1.pq: (250000, 99) за 14s, пик 5.26 ГБ



X_TRAIN: (3000000, 99), 1.12 ГБ
X_TEST:  (500000, 99), 0.19 ГБ
сборка: 15s, пик памяти 5.26 ГБ


## 3. Сверка train и test

Сравнивать средние по всем 99 колонкам нельзя: в train закодированные признаки
построены OOF (партиция `k` — статистика `0..k-1`, партиция 0 — нейтральный
ноль), а в test используется `FULL_ENCODER` по всем 12 партициям. Разница
средних у них частично объясняется схемой кодировки; вклад сдвига данных здесь не
измеряется.

Поэтому таблица ниже сравнивает только 26 не-кодируемых агрегатов: `n_records`,
`cold_start`, `share_open`, `share_noplan`, `nz_*`, `nzshare_*`, лестницу
просрочек и `last_util_nz`. `cold_start` — единственная колонка с ожидаемой
разницей: 250 000 из 3 000 000 train и 0 в test; это не дрейф, а индикатор
отсутствия предыдущих OOF-партиций. История у заявок партиции 0 есть; флаг не
означает «нет кредитной истории». Сдвиг закодированных признаков в этом ноутбуке отдельно не
измеряется: аудит выше считает только неизвестные и редкие уровни; одинаковыми
распределения закодированных колонок мы не считаем.


In [7]:
ENCODED_FEATURES = (
    [f"{prefix}_{column}" for prefix in ("mean", "last3", "last", "first")
     for column in RISK_COLUMNS]
    + ["RISK_mean", "RISK_mean_all_max", "RISK_max", "RISK_last3"]
    + ["PAYM_mean", "PAYM_last", "PAYM_first", "PAYM_wlast", "PAYM_recency"]
)
assert len(ENCODED_FEATURES) == 73
COMPARABLE_COLUMNS = [c for c in ALL_COLUMNS if c not in ENCODED_FEATURES]
assert len(COMPARABLE_COLUMNS) == 26

comparison = pd.DataFrame(
    {
        "среднее_train": X_TRAIN[COMPARABLE_COLUMNS].mean(),
        "среднее_test": X_TEST[COMPARABLE_COLUMNS].mean(),
        "std_train": X_TRAIN[COMPARABLE_COLUMNS].std(),
        "std_test": X_TEST[COMPARABLE_COLUMNS].std(),
    }
)
comparison["разница"] = comparison["среднее_test"] - comparison["среднее_train"]
last_part_mask = X_TRAIN.index >= 11 * 250_000
comparison["среднее_train_11"] = X_TRAIN.loc[last_part_mask, COMPARABLE_COLUMNS].mean()
comparison["разница_11_test"] = comparison["среднее_test"] - comparison["среднее_train_11"]
print(f"сравниваются только {len(COMPARABLE_COLUMNS)} не-кодируемых агрегатов")
for label, column in (
    ("top-15 по разнице средних train → test", "разница"),
    ("top-15 по разнице средних train-11 → test", "разница_11_test"),
):
    print(f"\n{label}:")
    print(
        comparison.reindex(comparison[column].abs().sort_values(ascending=False).index)
        .head(15)
        .round(4)
        .to_string()
    )
print(
    f"\nколонок, где средние совпали с точностью 0.01: "
    f"{int((comparison['разница'].abs() < 0.01).sum())} из {len(comparison)}"
)
print(f"колонок с наибольшей разницей train → test: {comparison['разница'].abs().max():.4f}")
print(
    "cold_start: ожидаемая разница 1/12 в train и 0 в test; "
    "в сравнении train-11 → test её нет."
)


сравниваются только 26 не-кодируемых агрегатов

top-15 по разнице средних train → test:
                  среднее_train  среднее_test  std_train   std_test  разница  среднее_train_11  разница_11_test
overdue_sev530          10.2540       12.4347    16.8197  20.103201   2.1807           11.0954           1.3393
overdue_ladder           6.6539        8.0971    11.0841  13.264500   1.4432            7.2125           0.8846
overdue_sev              3.6864        4.6626     7.1933   8.765600   0.9762            4.0121           0.6505
n_records                8.7209        9.4492     6.1762   6.579000   0.7283            9.8025          -0.3533
nz_over2limit            0.6095        0.8240     0.9512   1.117300   0.2146            0.7699           0.0541
nz_util                  2.6349        2.7825     1.8819   1.985400   0.1476            2.9931          -0.2107
nz_loans530              1.4633        1.6035     1.9732   2.137300   0.1403            1.6364          -0.0328
nz_maxover2limit

## 4. Обучение

Конфигурация `HistGradientBoostingClassifier` — та же, что проверялась в `03`:
`max_iter=300`, `learning_rate=0.08`, `min_samples_leaf=200`,
`l2_regularization=1.0`, `early_stopping=False`. Финальное обучение запускается
на всех 3 млн заявок. 300 итераций — фиксированный бюджет, а не проверка
сходимости: сходимость не проверялась.


In [8]:
Y_TRAIN = FLAGS.reindex(X_TRAIN.index).to_numpy()
if np.isnan(Y_TRAIN).any():
    raise ValueError("часть заявок train без метки")
assert len(Y_TRAIN) == 3_000_000
print(
    f"обучаем на {fmt_int(len(X_TRAIN))} заявках, положительных {fmt_int(int(Y_TRAIN.sum()))}"
)
print(f"доля положительного класса: {Y_TRAIN.mean():.4f}")

assert len(FINAL_COLUMNS) == FINAL_FEATURES
assert set(FINAL_COLUMNS) <= set(ALL_COLUMNS)
SELECTED_COLUMNS = [c for c in ALL_COLUMNS if c in set(FINAL_COLUMNS)]
assert SELECTED_COLUMNS == FINAL_COLUMNS
FINAL_DROP = sorted(set(ALL_COLUMNS) - set(FINAL_COLUMNS))
assert "cold_start" in SELECTED_COLUMNS
assert not any(c.startswith("rawlast") for c in SELECTED_COLUMNS)

first_columns = [c for c in SELECTED_COLUMNS if c.startswith("first_")]
print(f"\nвариант: {FINAL_VARIANT}")
print(f"признаков: {len(SELECTED_COLUMNS)} (FINAL_FEATURES = {FINAL_FEATURES})")
print(f"n_records: {'остался' if 'n_records' in SELECTED_COLUMNS else 'удалён'}")
print(
    f"first_*: {len(first_columns)}; PAYM_first: "
    f"{int('PAYM_first' in SELECTED_COLUMNS)}"
)
print(f"удалено колонок: {len(FINAL_DROP)}")

started = time.perf_counter()
MODEL = HistGradientBoostingClassifier(**HGB_PARAMS)
MODEL.fit(X_TRAIN[SELECTED_COLUMNS].to_numpy(np.float32), Y_TRAIN)
print(f"\nобучение завершено за {time.perf_counter() - started:.0f}s")
print(f"деревьев: {MODEL.n_iter_}, пик памяти {peak_memory_gb():.2f} ГБ")
assert MODEL.n_iter_ == HGB_PARAMS["max_iter"] == 300
print("n_iter_ == max_iter == 300: 300 — фиксированный бюджет, сходимость не проверялась.")
print(
    f"версии: scikit-learn {sklearn.__version__}, pandas {pd.__version__}, "
    f"numpy {np.__version__}, pyarrow {pyarrow.__version__}; SEED = {SEED}"
)


обучаем на 3 000 000 заявках, положительных 106 442
доля положительного класса: 0.0355

вариант: without_first
признаков: 79 (FINAL_FEATURES = 79)
n_records: остался
first_*: 0; PAYM_first: 0
удалено колонок: 20



обучение завершено за 78s
деревьев: 300, пик памяти 5.26 ГБ
n_iter_ == max_iter == 300: 300 — фиксированный бюджет, сходимость не проверялась.
версии: scikit-learn 1.9.1, pandas 3.0.6, numpy 2.5.4, pyarrow 26.0.0; SEED = 42


## 5. Предсказание и файл для отправки

Порядок `id` в `sample_submission.csv` задаёт Kaggle, поэтому предсказания
выравниваются по нему, а не по порядку партиций.


In [9]:
started = time.perf_counter()
SCORES = MODEL.predict_proba(X_TEST[SELECTED_COLUMNS].to_numpy(np.float32))[:, 1]
print(f"предсказание за {time.perf_counter() - started:.0f}s")
print(
    f"распределение score: минимум {SCORES.min():.5f}, "
    f"p1 {np.quantile(SCORES, 0.01):.5f}, медиана {float(np.median(SCORES)):.5f}, "
    f"p99 {np.quantile(SCORES, 0.99):.5f}, максимум {SCORES.max():.5f}"
)
print(f"средний score: {SCORES.mean():.5f}, доля выше 0.5: {(SCORES > 0.5).mean():.4f}")

SUBMISSION = pd.DataFrame({"id": X_TEST.index.to_numpy(), "score": SCORES})
SUBMISSION = SAMPLE[["id"]].merge(SUBMISSION, on="id", how="left")
assert len(SUBMISSION) == 500_000
assert SUBMISSION["id"].tolist() == SAMPLE["id"].tolist()
assert SUBMISSION["score"].notna().all()
assert SUBMISSION["id"].is_unique

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSION_PATH = PROCESSED_DIR / "submission.csv"
SUBMISSION.to_csv(SUBMISSION_PATH, index=False)
# pandas C-парсер по умолчанию не гарантирует round-trip; для точного
# сравнения score читаем с float_precision="round_trip".
CHECK = pd.read_csv(SUBMISSION_PATH, float_precision="round_trip")
assert CHECK.shape == SUBMISSION.shape
assert CHECK["score"].notna().all()
assert CHECK["id"].tolist() == SAMPLE["id"].tolist()
assert CHECK["id"].min() == 3_000_000
assert CHECK["id"].max() == 3_499_999
assert CHECK["id"].is_unique
assert np.array_equal(CHECK["score"].to_numpy(), SUBMISSION["score"].to_numpy())

EXPECTED_MD5 = "12f10bbd4ac49a518acf53ad8f5dfbbb"
MD5 = hashlib.md5(SUBMISSION_PATH.read_bytes()).hexdigest()
assert MD5 == EXPECTED_MD5, (MD5, EXPECTED_MD5)
print(f"\nзаписано {rel(SUBMISSION_PATH)}: {SUBMISSION.shape}")
print(SUBMISSION.head().to_string(index=False))
print(f"прочитано обратно: {CHECK.shape}, NaN нет, score совпадает точно")
print(f"md5: {MD5} (совпадает с EXPECTED_MD5)")
print(f"размер файла: {SUBMISSION_PATH.stat().st_size / 2**20:.2f} МиБ")


предсказание за 2s
распределение score: минимум 0.00147, p1 0.00342, медиана 0.02532, p99 0.17426, максимум 0.38602
средний score: 0.03807, доля выше 0.5: 0.0000



записано data/processed/submission.csv: (500000, 2)
     id    score
3000000 0.050120
3000001 0.056593
3000002 0.020551
3000003 0.036849
3000004 0.014714
прочитано обратно: (500000, 2), NaN нет, score совпадает точно
md5: 12f10bbd4ac49a518acf53ad8f5dfbbb (совпадает с EXPECTED_MD5)
размер файла: 13.58 МиБ


## 6. CV-only сравнение HGB и LightGBM

Это диагностика, а не выбор финальной модели: HGB уже обучен, submission уже
записан. Сравнение использует только `id < 2 000 000`; holdout-метки загружаются
для финального обучения, но в сравнении не используются, и holdout-AUC не
считается. Поле `holdout_labels_used=false` в артефакте относится только к
сравнению. Финальная модель остаётся HGB. В выводе «лучше в 4/6» относится к
LightGBM; средняя разница при этом отрицательна.


In [10]:
# CV-only: только первые 2M, holdout-метки не используются.
assert "MODEL" in globals()
del MODEL
_ = gc.collect()

assert len(SELECTED_COLUMNS) == 79
assert X_TRAIN.index.min() == 0
assert X_TRAIN.index.max() == 2_999_999
assert X_TRAIN.index.is_monotonic_increasing

CV_IDS = X_TRAIN.index.to_numpy()[:CV_END]
CV_X = X_TRAIN.iloc[:CV_END][SELECTED_COLUMNS].to_numpy(np.float32)
CV_Y = Y_TRAIN[:CV_END]
assert CV_IDS[0] == 0 and CV_IDS[-1] == CV_END - 1
assert CV_X.shape == (CV_END, 79)
assert not np.isnan(CV_Y).any()
assert np.isfinite(CV_X).all()


def classify_delta(delta_mean, delta_se, positive_folds):
    if delta_mean <= 0:
        return "keep_hgb"
    if delta_mean >= 0.001 and delta_mean > delta_se and positive_folds >= 5:
        return "lightgbm_candidate_for_new_validation"
    return "inconclusive_keep_hgb"


# Правило проверяется на синтетике до реальных чисел.
assert classify_delta(0.001, 0.0005, 5) == "lightgbm_candidate_for_new_validation"
assert classify_delta(0.002, 0.003, 6) == "inconclusive_keep_hgb"
assert classify_delta(0.002, 0.0005, 4) == "inconclusive_keep_hgb"
assert classify_delta(0.0005, 0.0001, 6) == "inconclusive_keep_hgb"
assert classify_delta(0.0, 0.0001, 3) == "keep_hgb"

FOLDS = list(range(FOLD_FIRST, FOLD_LAST + 1))
assert FOLDS == [2, 3, 4, 5, 6, 7]


def fold_masks(fold):
    low, high = fold * PART_SIZE, (fold + 1) * PART_SIZE
    train = CV_IDS < low
    valid = (CV_IDS >= low) & (CV_IDS < high)
    assert train.sum() == low
    assert valid.sum() == PART_SIZE
    assert not (train & valid).any()
    return train, valid


HGB_CV = []
for fold in FOLDS:
    train, valid = fold_masks(fold)
    started = time.perf_counter()
    model = HistGradientBoostingClassifier(**HGB_PARAMS)
    model.fit(CV_X[train], CV_Y[train])
    auc = roc_auc_score(CV_Y[valid], model.predict_proba(CV_X[valid])[:, 1])
    assert model.n_iter_ == HGB_PARAMS["max_iter"] == 300
    HGB_CV.append(
        {"fold": fold, "auc": float(auc), "seconds": float(time.perf_counter() - started)}
    )
    del model
    _ = gc.collect()

HGB_MEAN = float(np.mean([row["auc"] for row in HGB_CV]))
HGB_SD = float(np.std([row["auc"] for row in HGB_CV], ddof=0))
HGB_REFERENCE = float(SUMMARY["cv_mean"])
assert len(HGB_CV) == 6
assert abs(HGB_MEAN - 0.750282) <= 0.0005, (HGB_MEAN, CV_MEAN)
assert abs(HGB_MEAN - HGB_REFERENCE) <= 0.0005, (HGB_MEAN, HGB_REFERENCE)

LGBM_CV = []
for fold in FOLDS:
    train, valid = fold_masks(fold)
    started = time.perf_counter()
    model = lgb.LGBMClassifier(**LGBM_PARAMS)
    model.fit(CV_X[train], CV_Y[train])
    auc = roc_auc_score(CV_Y[valid], model.predict_proba(CV_X[valid])[:, 1])
    LGBM_CV.append(
        {"fold": fold, "auc": float(auc), "seconds": float(time.perf_counter() - started)}
    )
    del model
    _ = gc.collect()

LGBM_MEAN = float(np.mean([row["auc"] for row in LGBM_CV]))
LGBM_SD = float(np.std([row["auc"] for row in LGBM_CV], ddof=0))
DELTA = np.array([b["auc"] - a["auc"] for a, b in zip(HGB_CV, LGBM_CV)])
DELTA_MEAN = float(DELTA.mean())
DELTA_SD = float(DELTA.std(ddof=1))
DELTA_SE = float(DELTA_SD / np.sqrt(len(DELTA)))
POSITIVE_FOLDS = int((DELTA > 0).sum())
STATUS = classify_delta(DELTA_MEAN, DELTA_SE, POSITIVE_FOLDS)
assert len(DELTA) == 6

print(f"{'фолд':>4} {'HGB':>9} {'LightGBM':>9} {'LGBM-HGB':>11}")
for row in HGB_CV:
    lgbm_row = LGBM_CV[row["fold"] - FOLD_FIRST]
    print(
        f"{row['fold']:>4} {row['auc']:>9.6f} "
        f"{lgbm_row['auc']:>9.6f} "
        f"{lgbm_row['auc'] - row['auc']:>+11.6f}"
    )
print(f"HGB:      {HGB_MEAN:.6f} ± {HGB_SD:.6f} (ddof=0)")
print(f"LightGBM: {LGBM_MEAN:.6f} ± {LGBM_SD:.6f} (ddof=0)")
print(
    f"delta:    {DELTA_MEAN:+.6f} ± {DELTA_SE:.6f} (SE), "
    f"лучше в {POSITIVE_FOLDS}/6; статус: {STATUS}"
)

EXPLANATIONS = {
    "keep_hgb": "Средняя парная разница <= 0; HGB остаётся финальной моделью.",
    "inconclusive_keep_hgb": (
        "Средняя разница положительна, но не достигает заранее заданного "
        "порога устойчивости; HGB остаётся финальной моделью."
    ),
    "lightgbm_candidate_for_new_validation": (
        "LightGBM — кандидат на отдельную задачу валидации; финальная модель "
        "не заменяется автоматически."
    ),
}
assert STATUS in EXPLANATIONS
COMPARISON = {
    "notebook": "04-features-training.ipynb",
    "protocol": "cv_only_expanding_window",
    "variant": FINAL_VARIANT,
    "features": len(SELECTED_COLUMNS),
    "fit_end": CV_END,
    "folds": FOLDS,
    "part_size": PART_SIZE,
    "holdout_labels_used": False,
    "hgb": {
        "params": HGB_PARAMS,
        "fold_auc": [row["auc"] for row in HGB_CV],
        "mean_auc": HGB_MEAN,
        "sd_auc_ddof0": HGB_SD,
        "fit_seconds": [row["seconds"] for row in HGB_CV],
        "total_fit_seconds": float(sum(row["seconds"] for row in HGB_CV)),
        "reference_cv_mean": HGB_REFERENCE,
        "abs_delta_vs_reference": abs(HGB_MEAN - HGB_REFERENCE),
    },
    "lightgbm": {
        "version": lgb.__version__,
        "params": LGBM_PARAMS,
        "fold_auc": [row["auc"] for row in LGBM_CV],
        "mean_auc": LGBM_MEAN,
        "sd_auc_ddof0": LGBM_SD,
        "fit_seconds": [row["seconds"] for row in LGBM_CV],
        "total_fit_seconds": float(sum(row["seconds"] for row in LGBM_CV)),
    },
    "paired_delta_lightgbm_minus_hgb": {
        "fold_values": [float(value) for value in DELTA],
        "mean": DELTA_MEAN,
        "sd_ddof1": DELTA_SD,
        "se": DELTA_SE,
        "positive_folds": POSITIVE_FOLDS,
    },
    "decision": {
        "status": STATUS,
        "material_min_delta": 0.001,
        "min_positive_folds": 5,
        "replacement": False,
        "explanation": EXPLANATIONS[STATUS],
    },
}
assert COMPARISON["features"] == 79
assert COMPARISON["fit_end"] == 2_000_000
assert COMPARISON["holdout_labels_used"] is False
assert len(COMPARISON["hgb"]["fold_auc"]) == 6
assert len(COMPARISON["lightgbm"]["fold_auc"]) == 6
assert COMPARISON["decision"]["replacement"] is False

COMPARISON_PATH.parent.mkdir(parents=True, exist_ok=True)
COMPARISON_PATH.write_text(
    json.dumps(COMPARISON, ensure_ascii=False, indent=2), encoding="utf-8"
)
CHECK_COMPARISON = json.loads(COMPARISON_PATH.read_text(encoding="utf-8"))
assert CHECK_COMPARISON["decision"]["status"] == STATUS
assert CHECK_COMPARISON["folds"] == FOLDS
assert abs(
    sum(CHECK_COMPARISON["paired_delta_lightgbm_minus_hgb"]["fold_values"]) / 6
    - CHECK_COMPARISON["paired_delta_lightgbm_minus_hgb"]["mean"]
) <= 1e-6
print(f"записан {rel(COMPARISON_PATH)}")


фолд       HGB  LightGBM    LGBM-HGB
   2  0.757619  0.756671   -0.000948
   3  0.754585  0.752029   -0.002557
   4  0.749876  0.750336   +0.000460
   5  0.752231  0.752813   +0.000582
   6  0.746673  0.747075   +0.000402
   7  0.740708  0.740947   +0.000239
HGB:      0.750282 ± 0.005492 (ddof=0)
LightGBM: 0.749978 ± 0.004952 (ddof=0)
delta:    -0.000304 ± 0.000505 (SE), лучше в 4/6; статус: keep_hgb
записан data/processed/04_model_comparison.json


## Что получилось и чего ждать

| Параметр | Значение |
| --- | --- |
| Вариант | `without_first` |
| Признаков | 79 |
| Строк в финальном обучении | 3 000 000 |
| Оценка из `03-feature-analysis.ipynb` | **0.739142 ± 0.010409** |
| Метрика | ROC AUC |
| Внешний бейзлайн Kaggle | 0.75478 public / 0.74890 private |
| LightGBM, CV-only | 0.749978 ± 0.004952; Δ `−0.000304 ± 0.000505` (SE); 4/6; `keep_hgb` |

`±` — стандартное отклонение AUC по восьми блокам холдаута по 125 000 заявок
(`ddof=0`), а не ошибка среднего. Холдаут не участвовал в выборе признаков и
конфигурации; оценка из `03` относится к модели, обученной на 2 млн заявок; для
финальной модели на 3 млн она не является оценкой вне выборки. Финальное
обучение запускается на всех 3 млн заявок, `early_stopping=False`; 300 итераций —
фиксированный бюджет, сходимость не проверялась. Train-метрика всё равно
in-sample по построению и для оценки не используется.

Файл отправки: `data/processed/submission.csv`, 500 000 строк,
md5 `12f10bbd4ac49a518acf53ad8f5dfbbb`.

**Почему в test попадает нейтральный ноль.** Аудит test идёт против
`FULL_ENCODER = ENCODERS[12]` — словаря по всем 12 партициям train. Это
отличается от холдаут-аудита `03`, где проверка шла против `ENCODERS[8]`
(партиции `0..7`): там нужно было увидеть, что известно на момент решения,
здесь — что известно модели, предсказывающей test. Из 193 708 641 пары «строка
истории × кодируемая колонка» нейтральный ноль получают 5 019 пар (0.0026 %):
1 пара с уровнем за пределами таблицы и 5 018 пар на редких известных уровнях
(`counts < MIN_LEVEL_ROWS`). Это 4 916 различных строк истории: 1 строка с
неизвестным уровнем и 4 915 строк на редких уровнях. Основная масса —
`pre_over2limit` (4 239 строк); дальше `enc_loans_credit_status` (329),
`pre_pterm` (323), `pre_fterm` (120) и `pre_loans_credit_cost_rate` (7).
Единственный неизвестный уровень — `pre_loans_next_pay_summ = 7` при длине
таблицы 7, `id = 3464089`, `rn = 11`.

**Что здесь не проверено и не может быть проверено локально:**
- качество на самих тестовых заявках — меток нет;
- насколько вырос сдвиг распределений между последним отрезком train и test;
- горизонт наблюдения дефолта, из данных не восстанавливается.
